# Smoke tests de OP-06 `write_trips`

Este notebook prueba la operación pública `write_trips()` a nivel smoke test.

Objetivo:
- verificar que `write_trips()` persiste un `TripDataset` válido como artefacto formal;
- revisar el layout observable del bundle `.golondrina`;
- verificar side effects esperados sobre `metadata`;
- cubrir backend Parquet y backend Feather;
- probar fallas públicas relevantes sin entrar todavía a helper-level tests.

Convenciones:
- los tests usan `assert`;
- los artefactos se escriben en una carpeta visible junto al notebook;
- este notebook cubre solo OP-06 `write_trips`;
- no se prueba `read_trips` en este notebook.

## Bloque 0. Preparación

### 0.1 Imports generales

Qué prepara: imports básicos, filesystem, JSON, pandas y pyarrow para inspeccionar los artefactos escritos.

In [1]:
import copy
import json
import shutil
from pathlib import Path

import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.feather as feather
from pyarrow import types as patypes

### 0.2 Imports del módulo

Qué prepara: clases mínimas para construir un `TripDataset` sintético y la operación pública `write_trips`.

Importante: no se importa `read_trips`, porque este notebook cubre solo OP-06.

In [2]:
from pylondrina.schema import (
    DomainSpec,
    FieldSpec,
    TripSchema,
    TripSchemaEffective,
)

from pylondrina.datasets import TripDataset
from pylondrina.errors import ExportError, ValidationError

from pylondrina.io.trips import (
    write_trips,
    WriteTripsOptions,
)

### 0.3 Helpers de apoyo para test

Qué prepara: utilidades pequeñas para asserts, mensajes y lectura de sidecar.

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def get_issue_codes(issues):
    return [i.code if hasattr(i, "code") else i.get("code") for i in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró el issue {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente el issue {code}. Codes actuales: {codes}"


def load_sidecar(artifact_dir: Path) -> dict:
    sidecar_path = artifact_dir / "trips.metadata.json"
    assert sidecar_path.exists(), f"No existe sidecar: {sidecar_path}"
    return json.loads(sidecar_path.read_text(encoding="utf-8"))


def file_size(path: Path) -> int:
    assert path.exists(), f"No existe archivo: {path}"
    return path.stat().st_size

### 0.4 Configuración visual

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")
show_ok("Sección 0 cargada")

Imports OK
OK - Sección 0 cargada


## Bloque 1. Fixtures reutilizables mínimas

Qué prepara: factories pequeñas para crear un `TripDataset` sintético válido, con schema, schema efectivo, metadata y dominios categóricos suficientes para probar `write_trips`.

In [5]:
def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


def make_trip_schema(fields: list[FieldSpec], *, version: str = "1.1") -> TripSchema:
    return TripSchema(
        version=version,
        fields={f.name: f for f in fields},
        required=[f.name for f in fields if f.required],
        semantic_rules=None,
    )


def make_trip_schema_effective(
    *,
    dtype_effective: dict | None = None,
    overrides: dict | None = None,
    domains_effective: dict | None = None,
    temporal: dict | None = None,
    fields_effective: list | None = None,
) -> TripSchemaEffective:
    return TripSchemaEffective(
        dtype_effective=dtype_effective or {},
        overrides=overrides or {},
        domains_effective=domains_effective or {},
        temporal=temporal or {},
        fields_effective=fields_effective or [],
    )


def make_trip_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "movement_id": ["m1", "m2", "m3"],
            "trip_id": ["t1", "t2", "t3"],
            "movement_seq": [0, 0, 0],
            "user_id": ["u1", "u2", "u3"],
            "origin_latitude": [-33.45, -33.46, -33.47],
            "origin_longitude": [-70.66, -70.67, -70.68],
            "destination_latitude": [-33.41, -33.42, -33.43],
            "destination_longitude": [-70.61, -70.62, -70.63],
            "mode": ["bus", "metro", "bus"],
            "purpose": ["work", "study", "work"],
            "comment": ["a", "b", "c"],
            "trip_weight": [1.0, 2.5, 1.2],
        }
    )


def make_trip_schema_minimal() -> TripSchema:
    return make_trip_schema(
        [
            make_field("movement_id", "string", required=True),
            make_field("trip_id", "string", required=True),
            make_field("movement_seq", "int", required=True),
            make_field("user_id", "string", required=True),
            make_field("origin_latitude", "float", required=True),
            make_field("origin_longitude", "float", required=True),
            make_field("destination_latitude", "float", required=True),
            make_field("destination_longitude", "float", required=True),
            make_field(
                "mode",
                "categorical",
                required=False,
                domain=DomainSpec(values=["bus", "metro", "walk", "car"], extendable=True),
            ),
            make_field(
                "purpose",
                "categorical",
                required=False,
                domain=DomainSpec(values=["work", "study", "health"], extendable=True),
            ),
            make_field("comment", "string", required=False),
            make_field("trip_weight", "float", required=False),
        ]
    )


def make_trip_schema_effective_minimal() -> TripSchemaEffective:
    return make_trip_schema_effective(
        dtype_effective={
            "mode": "categorical",
            "purpose": "categorical",
            "trip_weight": "float",
        },
        domains_effective={
            "mode": {"values": ["bus", "metro", "walk", "car"]},
            "purpose": {"values": ["work", "study", "health"]},
        },
        temporal={"tier": "tier_3"},
        fields_effective=[
            "movement_id",
            "trip_id",
            "movement_seq",
            "user_id",
            "origin_latitude",
            "origin_longitude",
            "destination_latitude",
            "destination_longitude",
            "mode",
            "purpose",
            "comment",
            "trip_weight",
        ],
    )


def make_tripdataset(
    *,
    validated: bool = True,
    include_dataset_id: bool = True,
    include_artifact_id: bool = False,
) -> TripDataset:
    schema = make_trip_schema_minimal()
    schema_effective = make_trip_schema_effective_minimal()

    metadata = {
        "is_validated": validated,
        "events": [],
        "mappings": {
            "field_correspondence": {
                "movement_id": "movement_id_src",
                "mode": "mode_src",
            },
            "value_correspondence": {
                "mode": {
                    "micro": "bus",
                    "subte": "metro",
                }
            },
        },
        "domains_effective": copy.deepcopy(schema_effective.domains_effective),
        "temporal": {"tier": "tier_3"},
    }

    if include_dataset_id:
        metadata["dataset_id"] = "dset_test_001"
    if include_artifact_id:
        metadata["artifact_id"] = "art_test_001"

    provenance = {
        "source": {"name": "synthetic", "entity": "trips"},
        "ingestion": {"created_at_utc": "2026-04-04T00:00:00Z"},
    }

    return TripDataset(
        data=make_trip_df(),
        schema=schema,
        schema_version=schema.version,
        provenance=provenance,
        field_correspondence={"movement_id": "movement_id_src", "mode": "mode_src"},
        value_correspondence={"mode": {"micro": "bus", "subte": "metro"}},
        metadata=metadata,
        schema_effective=schema_effective,
    )

## Bloque 2. Carpeta visible para artefactos de smoke tests

Qué prepara: una carpeta local, junto al notebook, para inspeccionar manualmente los bundles escritos por `write_trips`.

La carpeta se reinicia al ejecutar este bloque.

In [6]:
SMOKE_ROOT = Path("./tmp_op06_write_trips_smoke")


def reset_smoke_root() -> Path:
    if SMOKE_ROOT.exists():
        shutil.rmtree(SMOKE_ROOT)
    SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
    return SMOKE_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = SMOKE_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


root = reset_smoke_root()
print("SMOKE_ROOT =", root.resolve())
show_ok("Bloque 2 - carpeta visible preparada")

SMOKE_ROOT = C:\projects\pylondrina\notebooks\testing\io_trips\tmp_op06_write_trips_smoke
OK - Bloque 2 - carpeta visible preparada


## Bloque 3. Smoke tests de `write_trips`

### Test 3.1 - smoke test de `write_trips` happy path con Parquet y normalización `.golondrina`

Qué prueba: escritura formal correcta con dataset validado, usando backend Parquet. Verifica:

- `OperationReport.ok`;
- ausencia de issues;
- creación del bundle `.golondrina`;
- existencia de `trips.parquet` y `trips.metadata.json`;
- summary y parameters;
- side effects esperados en `trips.metadata`;
- sidecar consistente;
- no mutación de `trips.data`.

In [7]:
case_dir = make_case_dir("case_01_write_parquet_happy")
artifact_dir = case_dir / "artifact_write_parquet_happy"
true_artifact_dir = case_dir / "artifact_write_parquet_happy.golondrina"

trips = make_tripdataset(validated=True)
data_before = trips.data.copy(deep=True)
metadata_before = copy.deepcopy(trips.metadata)

report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
    ),
)

assert report.ok is True
assert len(report.issues) == 0

# Layout formal persistido
assert true_artifact_dir.exists()
assert true_artifact_dir.is_dir()
assert (true_artifact_dir / "trips.parquet").exists()
assert (true_artifact_dir / "trips.metadata.json").exists()
assert not (true_artifact_dir / "trips.feather").exists()

# Summary y parameters
assert report.summary["n_rows"] == len(trips.data)
assert Path(report.summary["path"]) == true_artifact_dir
assert report.summary["artifact_id"] == trips.metadata["artifact_id"]
assert report.summary["dataset_id"] == trips.metadata["dataset_id"]
assert report.summary["dataset_id_status"] == "preserved"
assert report.summary["storage_format"] == "parquet"
assert report.summary["files_written"] == ["trips.parquet", "trips.metadata.json"]

assert report.parameters["storage_format"] == "parquet"
assert report.parameters["mode"] == "error_if_exists"
assert report.parameters["require_validated"] is True
assert report.parameters["parquet_compression"] == "snappy"
assert report.parameters["normalize_artifact_dir"] is True

# Side effects en memoria
assert "dataset_id" in trips.metadata
assert "artifact_id" in trips.metadata
assert trips.metadata["dataset_id"] == metadata_before["dataset_id"]
assert trips.metadata["is_validated"] is True
assert len(trips.metadata["events"]) == len(metadata_before["events"]) + 1
assert trips.metadata["events"][-1]["op"] == "write_trips"

# El dataframe no debe mutar
pd.testing.assert_frame_equal(trips.data, data_before)

# Sidecar cargable e íntegro
sidecar = load_sidecar(true_artifact_dir)

assert sidecar["dataset_type"] == "trips"
assert sidecar["format"] == "golondrina"
assert sidecar["layout_version"] == "1.1"

assert sidecar["storage"]["format"] == "parquet"
assert sidecar["storage"]["options"]["compression"] == "snappy"

assert sidecar["files"]["data"] == "trips.parquet"
assert sidecar["files"]["metadata"] == "trips.metadata.json"

assert sidecar["dataset_id"] == trips.metadata["dataset_id"]
assert sidecar["artifact_id"] == trips.metadata["artifact_id"]

assert "schema" in sidecar
assert "schema_effective" in sidecar
assert "provenance" in sidecar
assert "metadata" in sidecar

assert sidecar["metadata"]["dataset_id"] == trips.metadata["dataset_id"]
assert sidecar["metadata"]["artifact_id"] == trips.metadata["artifact_id"]
assert sidecar["metadata"]["events"][-1]["op"] == "write_trips"

display(report)
show_ok("Test 3.1 - write_trips happy path Parquet")

OperationReport(ok=True, issues=[], summary={'n_rows': 3, 'files_written': ['trips.parquet', 'trips.metadata.json'], 'path': 'tmp_op06_write_trips_smoke\\case_01_write_parquet_happy\\artifact_write_parquet_happy.golondrina', 'dataset_id': 'dset_test_001', 'artifact_id': 'art_88fdfb45-42ef-4bb9-b7ee-6986c35fa462', 'dataset_id_status': 'preserved', 'storage_format': 'parquet'}, parameters={'path': 'tmp_op06_write_trips_smoke\\case_01_write_parquet_happy\\artifact_write_parquet_happy.golondrina', 'mode': 'error_if_exists', 'require_validated': True, 'storage_format': 'parquet', 'parquet_compression': 'snappy', 'feather_compression': 'lz4', 'normalize_artifact_dir': True})

OK - Test 3.1 - write_trips happy path Parquet


### Test 3.2 - smoke test de `write_trips` happy path con Feather

Qué prueba: escritura formal correcta con backend Feather. Este test es necesario porque la implementación vigente de OP-06 ya no es Parquet-only.

Verifica:

- archivo `trips.feather`;
- ausencia de `trips.parquet`;
- sidecar con `storage.format = "feather"`;
- opciones Feather v2;
- summary y parameters coherentes.

In [8]:
case_dir = make_case_dir("case_02_write_feather_happy")
artifact_dir = case_dir / "artifact_write_feather_happy"

trips = make_tripdataset(validated=True)
data_before = trips.data.copy(deep=True)

report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="feather",
        feather_compression="lz4",
        normalize_artifact_dir=False,
    ),
)

assert report.ok is True
assert len(report.issues) == 0

# Layout formal persistido
assert artifact_dir.exists()
assert artifact_dir.is_dir()
assert (artifact_dir / "trips.feather").exists()
assert (artifact_dir / "trips.metadata.json").exists()
assert not (artifact_dir / "trips.parquet").exists()

# Summary y parameters
assert report.summary["n_rows"] == len(trips.data)
assert Path(report.summary["path"]) == artifact_dir
assert report.summary["artifact_id"] == trips.metadata["artifact_id"]
assert report.summary["dataset_id"] == trips.metadata["dataset_id"]
assert report.summary["dataset_id_status"] == "preserved"
assert report.summary["storage_format"] == "feather"
assert report.summary["files_written"] == ["trips.feather", "trips.metadata.json"]

assert report.parameters["storage_format"] == "feather"
assert report.parameters["mode"] == "error_if_exists"
assert report.parameters["require_validated"] is True
assert report.parameters["feather_compression"] == "lz4"
assert report.parameters["normalize_artifact_dir"] is False

# Side effects en memoria
assert trips.metadata["is_validated"] is True
assert trips.metadata["events"][-1]["op"] == "write_trips"

# El dataframe no debe mutar
pd.testing.assert_frame_equal(trips.data, data_before)

# Sidecar backend-aware
sidecar = load_sidecar(artifact_dir)

assert sidecar["storage"]["format"] == "feather"
assert sidecar["storage"]["options"]["compression"] == "lz4"
assert sidecar["storage"]["options"]["version"] == 2

assert sidecar["files"]["data"] == "trips.feather"
assert sidecar["files"]["metadata"] == "trips.metadata.json"

assert sidecar["metadata"]["artifact_id"] == trips.metadata["artifact_id"]
assert sidecar["metadata"]["events"][-1]["op"] == "write_trips"

# Inspección mínima de que Feather es legible por Arrow
table = feather.read_table(artifact_dir / "trips.feather")
assert table.num_rows == len(trips.data)
assert table.schema.names == list(trips.data.columns)

display(report)
show_ok("Test 3.2 - write_trips happy path Feather")

OperationReport(ok=True, issues=[], summary={'n_rows': 3, 'files_written': ['trips.feather', 'trips.metadata.json'], 'path': 'tmp_op06_write_trips_smoke\\case_02_write_feather_happy\\artifact_write_feather_happy', 'dataset_id': 'dset_test_001', 'artifact_id': 'art_8715a991-26c5-425a-a903-5bdff661435b', 'dataset_id_status': 'preserved', 'storage_format': 'feather'}, parameters={'path': 'tmp_op06_write_trips_smoke\\case_02_write_feather_happy\\artifact_write_feather_happy', 'mode': 'error_if_exists', 'require_validated': True, 'storage_format': 'feather', 'parquet_compression': 'snappy', 'feather_compression': 'lz4', 'normalize_artifact_dir': False})

OK - Test 3.2 - write_trips happy path Feather


### Test 3.3 - smoke test de categóricos persistidos en Parquet

Qué prueba: `write_trips()` escribe columnas categóricas de forma compatible con Arrow/Parquet. Se inspecciona que `mode` y `purpose` queden con dictionary encoding a nivel Parquet.

Este test viene del smoke test original, pero queda restringido a OP-06.

In [9]:
case_dir = make_case_dir("case_03_parquet_categorical_encoding")
artifact_dir = case_dir / "artifact_parquet_categorical"

# Dataset más grande para que el archivo sea más representativo.
trips = make_tripdataset(validated=True)
df_big = pd.concat([trips.data] * 3000, ignore_index=True)

trips_big = make_tripdataset(validated=True)
trips_big.data = df_big

report = write_trips(
    trips_big,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
    ),
)

assert report.ok is True
assert len(report.issues) == 0

parquet_path = artifact_dir / "trips.parquet"
assert parquet_path.exists()

parquet_file = pq.ParquetFile(parquet_path)
try:
    names = parquet_file.schema_arrow.names

    for col_name in ["mode", "purpose"]:
        assert col_name in names
        col_idx = names.index(col_name)

        encodings = {
            str(enc).upper()
            for enc in parquet_file.metadata.row_group(0).column(col_idx).encodings
        }

        assert any("DICTIONARY" in enc for enc in encodings), (
            f"{col_name} no quedó con dictionary encoding: {encodings}"
        )
finally:
    parquet_file.close()

print("parquet_size_bytes =", file_size(parquet_path))
display(report.summary)
show_ok("Test 3.3 - categóricos persistidos en Parquet")

parquet_size_bytes = 8882


{'n_rows': 9000,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'tmp_op06_write_trips_smoke\\case_03_parquet_categorical_encoding\\artifact_parquet_categorical',
 'dataset_id': 'dset_test_001',
 'artifact_id': 'art_6464c17a-ee14-4987-869a-0a06ed74ed8c',
 'dataset_id_status': 'preserved',
 'storage_format': 'parquet'}

OK - Test 3.3 - categóricos persistidos en Parquet


### Test 3.4 - smoke test de categóricos persistidos en Feather

Qué prueba: `write_trips()` escribe columnas categóricas de forma compatible con Arrow/Feather v2. Se inspecciona que `mode` y `purpose` queden como columnas dictionary en la tabla Arrow.

In [10]:
case_dir = make_case_dir("case_04_feather_categorical_encoding")
artifact_dir = case_dir / "artifact_feather_categorical"

trips = make_tripdataset(validated=True)
df_big = pd.concat([trips.data] * 3000, ignore_index=True)

trips_big = make_tripdataset(validated=True)
trips_big.data = df_big

report = write_trips(
    trips_big,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="feather",
        feather_compression="lz4",
        normalize_artifact_dir=False,
    ),
)

assert report.ok is True
assert len(report.issues) == 0

feather_path = artifact_dir / "trips.feather"
assert feather_path.exists()

table = feather.read_table(feather_path)
assert table.num_rows == len(trips_big.data)
assert table.schema.names == list(trips_big.data.columns)

for col_name in ["mode", "purpose"]:
    arrow_type = table.schema.field(col_name).type
    assert patypes.is_dictionary(arrow_type), (
        f"{col_name} no quedó como dictionary en Feather: {arrow_type}"
    )

print("feather_size_bytes =", file_size(feather_path))
display(report.summary)
show_ok("Test 3.4 - categóricos persistidos en Feather")

feather_size_bytes = 153314


{'n_rows': 9000,
 'files_written': ['trips.feather', 'trips.metadata.json'],
 'path': 'tmp_op06_write_trips_smoke\\case_04_feather_categorical_encoding\\artifact_feather_categorical',
 'dataset_id': 'dset_test_001',
 'artifact_id': 'art_008a8dd2-ad73-47b9-8c74-dd7749ac7eab',
 'dataset_id_status': 'preserved',
 'storage_format': 'feather'}

OK - Test 3.4 - categóricos persistidos en Feather


### Test 3.5 - smoke test de creación de `dataset_id` faltante

Qué prueba: si el `TripDataset` no trae `metadata["dataset_id"]`, `write_trips()` genera uno nuevo, registra issue informativo y lo persiste en metadata y sidecar.

Este comportamiento pertenece a OP-06 porque la escritura formal cierra identidad lógica cuando falta.

In [11]:
case_dir = make_case_dir("case_05_dataset_id_created")
artifact_dir = case_dir / "artifact_dataset_id_created"

trips = make_tripdataset(
    validated=True,
    include_dataset_id=False,
)

assert "dataset_id" not in trips.metadata

report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
    ),
)

assert report.ok is True
assert_issue_present(report.issues, "WRT.METADATA.DATASET_ID_CREATED")

assert "dataset_id" in trips.metadata
assert isinstance(trips.metadata["dataset_id"], str)
assert trips.metadata["dataset_id"].startswith("dset_")

assert "artifact_id" in trips.metadata
assert isinstance(trips.metadata["artifact_id"], str)
assert trips.metadata["artifact_id"].startswith("art_")

assert report.summary["dataset_id"] == trips.metadata["dataset_id"]
assert report.summary["dataset_id_status"] == "created"

sidecar = load_sidecar(artifact_dir)
assert sidecar["dataset_id"] == trips.metadata["dataset_id"]
assert sidecar["metadata"]["dataset_id"] == trips.metadata["dataset_id"]
assert sidecar["artifact_id"] == trips.metadata["artifact_id"]

display(report.issues)
display(report.summary)
show_ok("Test 3.5 - dataset_id faltante creado por write_trips")

[Issue(level='info', code='WRT.METADATA.DATASET_ID_CREATED', message="Se generó dataset_id para poder persistir el artefacto de trips: 'dset_f3e81015-ea94-4612-9d8e-372a65848a43'.", field=None, source_field=None, row_count=None, details={'dataset_id': 'dset_f3e81015-ea94-4612-9d8e-372a65848a43', 'generator': 'uuid4', 'stored_in': 'metadata.dataset_id'})]

{'n_rows': 3,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'tmp_op06_write_trips_smoke\\case_05_dataset_id_created\\artifact_dataset_id_created',
 'dataset_id': 'dset_f3e81015-ea94-4612-9d8e-372a65848a43',
 'artifact_id': 'art_426293d2-4d17-447d-a131-6b7d3c5d748c',
 'dataset_id_status': 'created',
 'storage_format': 'parquet'}

OK - Test 3.5 - dataset_id faltante creado por write_trips


### Test 3.6 - smoke test de `mode="overwrite"`

Qué prueba: `write_trips()` puede reemplazar un artefacto existente cuando se usa `mode="overwrite"`. Además, verifica que cada escritura exitosa genera un nuevo `artifact_id`.

In [12]:
case_dir = make_case_dir("case_06_overwrite")
artifact_dir = case_dir / "artifact_overwrite"

trips = make_tripdataset(validated=True)

first_report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
    ),
)

first_artifact_id = trips.metadata["artifact_id"]

assert first_report.ok is True
assert artifact_dir.exists()
assert (artifact_dir / "trips.parquet").exists()
assert (artifact_dir / "trips.metadata.json").exists()

# Archivo residual que debería desaparecer al sobrescribir.
(artifact_dir / "old_residual.txt").write_text("old", encoding="utf-8")
assert (artifact_dir / "old_residual.txt").exists()

second_report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="overwrite",
        require_validated=True,
        storage_format="feather",
        feather_compression="lz4",
        normalize_artifact_dir=False,
    ),
)

second_artifact_id = trips.metadata["artifact_id"]

assert second_report.ok is True
assert second_artifact_id != first_artifact_id

assert artifact_dir.exists()
assert not (artifact_dir / "old_residual.txt").exists()

# El segundo write fue Feather, por tanto el layout final debe reflejar eso.
assert not (artifact_dir / "trips.parquet").exists()
assert (artifact_dir / "trips.feather").exists()
assert (artifact_dir / "trips.metadata.json").exists()

sidecar = load_sidecar(artifact_dir)
assert sidecar["storage"]["format"] == "feather"
assert sidecar["files"]["data"] == "trips.feather"
assert sidecar["artifact_id"] == second_artifact_id

display(first_report.summary)
display(second_report.summary)
show_ok("Test 3.6 - mode overwrite")

{'n_rows': 3,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'tmp_op06_write_trips_smoke\\case_06_overwrite\\artifact_overwrite',
 'dataset_id': 'dset_test_001',
 'artifact_id': 'art_9f61a22b-09bf-4102-8605-4c411ed9a78d',
 'dataset_id_status': 'preserved',
 'storage_format': 'parquet'}

{'n_rows': 3,
 'files_written': ['trips.feather', 'trips.metadata.json'],
 'path': 'tmp_op06_write_trips_smoke\\case_06_overwrite\\artifact_overwrite',
 'dataset_id': 'dset_test_001',
 'artifact_id': 'art_4a01ed72-70d9-4333-ba4c-4ce58a2f93da',
 'dataset_id_status': 'preserved',
 'storage_format': 'feather'}

OK - Test 3.6 - mode overwrite


### Test 3.7 - smoke test fatal por dataset no validado

Qué prueba: si `require_validated=True` y el dataset no está validado, `write_trips()` aborta con `ValidationError` y no materializa el artefacto.

In [13]:
case_dir = make_case_dir("case_07_fatal_not_validated")
artifact_dir = case_dir / "artifact_not_validated"

trips = make_tripdataset(validated=False)

raised = None
try:
    write_trips(
        trips,
        artifact_dir,
        options=WriteTripsOptions(
            mode="error_if_exists",
            require_validated=True,
            storage_format="parquet",
            parquet_compression="snappy",
            normalize_artifact_dir=False,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ValidationError)
assert not artifact_dir.exists()

display(raised)
show_ok("Test 3.7 - fatal por dataset no validado")

ValidationError(message="write_trips requiere un dataset validado, pero metadata['is_validated']=False.", code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', details={'require_validated': True, 'validated_flag': False, 'path': 'tmp_op06_write_trips_smoke\\case_07_fatal_not_validated\\artifact_not_validated', 'action': 'abort'}, issue=Issue(level='error', code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', message="write_trips requiere un dataset validado, pero metadata['is_validated']=False.", field=None, source_field=None, row_count=None, details={'require_validated': True, 'validated_flag': False, 'path': 'tmp_op06_write_trips_smoke\\case_07_fatal_not_validated\\artifact_not_validated', 'action': 'abort'}), issues=(Issue(level='error', code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', message="write_trips requiere un dataset validado, pero metadata['is_validated']=False.", field=None, source_field=None, row_count=None, details={'require_validated': True, 'validated_flag': False, 'path': 'tmp_op06_wr

OK - Test 3.7 - fatal por dataset no validado


### Test 3.8 - smoke test fatal por destino existente con `error_if_exists`

Qué prueba: si el destino ya existe y se usa `mode="error_if_exists"`, `write_trips()` aborta sin reemplazar el contenido existente.

In [14]:
case_dir = make_case_dir("case_08_fatal_destination_exists")
artifact_dir = case_dir / "artifact_existing"

artifact_dir.mkdir(parents=True, exist_ok=True)
sentinel = artifact_dir / "sentinel.txt"
sentinel.write_text("do not delete", encoding="utf-8")

trips = make_tripdataset(validated=True)

raised = None
try:
    write_trips(
        trips,
        artifact_dir,
        options=WriteTripsOptions(
            mode="error_if_exists",
            require_validated=True,
            storage_format="parquet",
            parquet_compression="snappy",
            normalize_artifact_dir=False,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)

# El contenido previo no debe eliminarse en modo error_if_exists.
assert artifact_dir.exists()
assert sentinel.exists()
assert sentinel.read_text(encoding="utf-8") == "do not delete"

display(raised)
show_ok("Test 3.8 - fatal por destino existente")

ExportError(message="El destino formal ya existe en 'tmp_op06_write_trips_smoke\\\\case_08_fatal_destination_exists\\\\artifact_existing' y mode='error_if_exists' no permite sobrescribirlo.", code='WRT.DEST.ALREADY_EXISTS', details={'path': 'tmp_op06_write_trips_smoke\\case_08_fatal_destination_exists\\artifact_existing', 'resolved_path': 'tmp_op06_write_trips_smoke\\case_08_fatal_destination_exists\\artifact_existing', 'mode': 'error_if_exists', 'files_present_sample': ['sentinel.txt'], 'files_present_total': 1, 'action': 'abort'}, issue=Issue(level='error', code='WRT.DEST.ALREADY_EXISTS', message="El destino formal ya existe en 'tmp_op06_write_trips_smoke\\\\case_08_fatal_destination_exists\\\\artifact_existing' y mode='error_if_exists' no permite sobrescribirlo.", field=None, source_field=None, row_count=None, details={'path': 'tmp_op06_write_trips_smoke\\case_08_fatal_destination_exists\\artifact_existing', 'resolved_path': 'tmp_op06_write_trips_smoke\\case_08_fatal_destination_exi

OK - Test 3.8 - fatal por destino existente


### Test 3.9 - smoke test de `require_validated=False`

Qué prueba: `write_trips()` permite persistir un dataset no validado cuando el usuario desactiva explícitamente la precondición. La operación no certifica el dataset, solo conserva `metadata["is_validated"] = False` en el sidecar y en memoria.

In [15]:
case_dir = make_case_dir("case_09_require_validated_false")
artifact_dir = case_dir / "artifact_require_validated_false"

trips = make_tripdataset(validated=False)

report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=False,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
    ),
)

assert report.ok is True
assert_issue_absent(report.issues, "WRT.VALIDATION.REQUIRED_NOT_VALIDATED")

assert artifact_dir.exists()
assert (artifact_dir / "trips.parquet").exists()
assert (artifact_dir / "trips.metadata.json").exists()

assert trips.metadata["is_validated"] is False
assert trips.metadata["events"][-1]["op"] == "write_trips"

sidecar = load_sidecar(artifact_dir)
assert sidecar["metadata"]["is_validated"] is False
assert sidecar["metadata"]["events"][-1]["op"] == "write_trips"

assert report.parameters["require_validated"] is False

display(report.summary)
show_ok("Test 3.9 - require_validated=False")

{'n_rows': 3,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'tmp_op06_write_trips_smoke\\case_09_require_validated_false\\artifact_require_validated_false',
 'dataset_id': 'dset_test_001',
 'artifact_id': 'art_a6583592-bda2-4d7f-a710-f82dc6652cc8',
 'dataset_id_status': 'preserved',
 'storage_format': 'parquet'}

OK - Test 3.9 - require_validated=False
